In [4]:
import pandas as pd
from pathlib import Path

# --- パス設定 ---
入力ディレクトリパス = '/Users/muna/Hana_research/data/raw/AllLab'
出力ファイル名 = 'Cre2_2026.csv'
出力ファイルフルパス = Path.home() / 'Desktop' / 出力ファイル名

input_dir_path = Path(入力ディレクトリパス)
csv_files_list = list(input_dir_path.glob("*.csv"))
print(f"{len(csv_files_list)} 件の CSV を処理します")

all_result_dataframes = []

for file in csv_files_list:
    print(f"処理中：{file.name}")
    try:
        df = pd.read_csv(file, encoding='utf_8_sig', dtype=str, index_col=0)  # 先頭列をindexとして読む
    except Exception as e:
        print(f"  読み込み失敗：{e}")
        continue

    required_columns = ['検査日', '検査項目', '検査値']
    if not all(col in df.columns for col in required_columns):
        print(f"  列不足でスキップ：{[c for c in required_columns if c not in df.columns]}")
        continue

    try:
        # デバッグ：最初のファイルだけ検査項目の一覧を表示
        if len(all_result_dataframes) == 0:
            print(f"  検査項目サンプル：{df['検査項目'].unique()[:10]}")

        # 1. 検査日フィルタ
        df['検査日_dt'] = pd.to_datetime(df['検査日'], errors='coerce')
        df = df[df['検査日_dt'].dt.year >= 2026].copy()
        if df.empty:
            print("  → 2026年以降のデータなし")
            continue

        # 2. CRE抽出（全角・半角・前後スペース対応）
        df['検査項目_norm'] = df['検査項目'].str.strip().str.replace('　', '').str.replace(' ', '')
        # 全角→半角変換
        df['検査項目_norm'] = df['検査項目_norm'].str.translate(
            str.maketrans('ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺ',
                          'ABCDEFGHIJKLMNOPQRSTUVWXYZ')
        )
        df = df[df['検査項目_norm'] == 'CRE'].copy()
        if df.empty:
            print("  → CRE該当なし")
            continue

        # 3. 検査値 >= 2.0
        df['検査値'] = pd.to_numeric(df['検査値'], errors='coerce')
        df = df[df['検査値'] >= 2.0].copy()
        if df.empty:
            print("  → 検査値2.0以上なし")
            continue

        df = df.drop(columns=['検査日_dt', '検査項目_norm'])
        print(f"  → {len(df)} 行を抽出")
        all_result_dataframes.append(df.reset_index(drop=True))

    except Exception as e:
        print(f"  処理エラー：{e}")
        continue

# --- 統合・出力 ---
if all_result_dataframes:
    final_csv = (
        pd.concat(all_result_dataframes, ignore_index=True)
        .sort_values(['検査日', '患者ID'])
        .drop_duplicates(subset=['検査日', '患者ID'], keep='first')  # 追加
        .reset_index(drop=True)
    )
    print(f"\n合計 {len(final_csv)} 行")
    final_csv.to_csv(出力ファイルフルパス, index=False, encoding='utf_8_sig')
    print(f"保存完了：{出力ファイルフルパス}")
else:
    print("条件を満たすデータが見つかりませんでした")

2922 件の CSV を処理します
処理中：DS180208.csv
  検査項目サンプル：<StringArray>
['総蛋白', 'ＧＯＴ', 'ＧＰＴ', 'ＬＤＨ', 'ＣＰＫ', 'γ－ＧＴＰ', 'ＡＬＰ', 'ＣＨＥ', 'Ｔ－ＢＩＬ', '尿素窒素']
Length: 10, dtype: str
  → 2026年以降のデータなし
処理中：DS151121.csv
  検査項目サンプル：<StringArray>
['総蛋白', 'ＧＯＴ', 'ＧＰＴ', 'ＬＤＨ', 'ＣＰＫ', 'γ－ＧＴＰ', 'ＡＬＰ', 'Ｔ－ＢＩＬ', '尿素窒素', 'ＣＲＥ']
Length: 10, dtype: str
  → 2026年以降のデータなし
処理中：DS260422.csv
  検査項目サンプル：<StringArray>
['総蛋白', 'アルブミン', 'ＧＯＴ', 'ＧＰＴ', 'ＬＤ／ＩＦ', 'ＣＰＫ', 'γ－ＧＴＰ', 'Ｔ－ＢＩＬ', '尿素窒素',
 'ＣＲＥ']
Length: 10, dtype: str
  → 3 行を抽出
処理中：DS230215.csv
  → 2026年以降のデータなし
処理中：DS230201.csv
  → 2026年以降のデータなし
処理中：DS160426 - コピー (2).csv
  → 2026年以降のデータなし
処理中：DS231123.csv
  → 2026年以降のデータなし
処理中：DS180220.csv
  → 2026年以降のデータなし
処理中：DS200315.csv
  → 2026年以降のデータなし
処理中：DS210106.csv
  → 2026年以降のデータなし
処理中：DS240731.csv
  → 2026年以降のデータなし
処理中：DS211218.csv
  → 2026年以降のデータなし
処理中：DS250522.csv
  → 2026年以降のデータなし
処理中：DS161021.csv
  → 2026年以降のデータなし
処理中：DS171226.csv
  → 2026年以降のデータなし
処理中：DS200301.csv
  → 2026年以降のデータなし
処理中：DS240725.csv
  → 2026年以降のデータなし
処理中：DS